<table border=0 width="100%"><tr><td><p align="left"><font size=5><B>数据分析与可视化</B></font></p></td><td><font size=3><B>Exploratory Data Analysis(郑海超)</B></font></td></tr></table>

### 下载上证指数日线

先下载再分析。上证指数代码 `000001.SH`，从 2001 年取到 2026 年 10 月 10 日。接口单次有行数上限，按年取再拼起来，并落盘。

In [3]:
import tushare as ts
import pandas as pd

token = "5c426f5af978fd42fe0f81fe7e60c02ada33d6c79347b61ccb44f521"
ts.set_token(token)
pro = ts.pro_api()

frames = []
for year in range(2001, 2027):
    end = "20261010" if year == 2026 else f"{year}1231"
    part = pro.index_daily(ts_code="000001.SH", start_date=f"{year}0101", end_date=end)
    frames.append(part)

raw = pd.concat(frames, ignore_index=True)
raw["trade_date"] = pd.to_datetime(raw["trade_date"].astype(str))
raw = raw.sort_values("trade_date")
raw.to_csv("sh_index_daily.csv", index=False)
raw["trade_date"].min(), raw["trade_date"].max(), raw.shape

(6241, 11)

In [ ]:
df = pd.read_csv("sh_index_daily.csv")
df.head()

### 读入收盘价

这一格只留下日期和收盘价。问：这格得到了什么？

In [ ]:
close = df[["trade_date", "close"]].copy()
close.head()

### 日期转为 datetime，设为索引并排序

`trade_date` 现在多半是 `20011231` 这样的数字或字符串。转成日期、放到索引上，再按时间从早到晚排。不排序的话，下一格的收益率会用错相邻两天。

In [ ]:
close["trade_date"] = pd.to_datetime(close["trade_date"].astype(str))
close = close.set_index("trade_date").sort_index()
close.head()

### 简单收益率和对数收益率

简单收益率 \(R=P_t/P_{t-1}-1\)，用 `pct_change`。对数收益率 \(r=\ln(P_t/P_{t-1})=\ln(1+R)\)。

\(R\) 接近 0 时，\(\ln(1+R)=R-R^2/2+\cdots\approx R\)。并排看几天：平常日几乎一样，涨跌很大时拉开。后面分组仍用简单收益率。

In [ ]:
import numpy as np

close["R"] = close["close"].pct_change()
close["log_return"] = np.log(close["close"] / close["close"].shift(1))
close.loc[["2001-06-13", "2024-09-30"], ["close", "R", "log_return"]]

### 给每个交易日打窗口标签

新加一列 `窗口`。每年 10 月 1 日之前的 5 个交易日写成「节前」，之后的 5 个交易日写成「节后」，其余写成「普通日」。10 月 1 日本身不是交易日，不占一行。

In [2]:
import pandas as pd

def label_window(dates):
    dates = pd.DatetimeIndex(pd.to_datetime(dates)).sort_values()
    window = pd.Series("普通日", index=dates, name="窗口")
    for year in range(dates.min().year, dates.max().year + 1):
        oct1 = pd.Timestamp(year=year, month=10, day=1)
        window.loc[dates[dates < oct1][-5:]] = "节前"
        window.loc[dates[dates > oct1][:5]] = "节后"
    return window

# 一小段交易日：休市日没放进来，所以「之后 5 个」就是节后 5 个交易日
demo = pd.to_datetime([
    "2024-09-20", "2024-09-23", "2024-09-24", "2024-09-25", "2024-09-26",
    "2024-09-27", "2024-09-30",
    "2024-10-08", "2024-10-09", "2024-10-10", "2024-10-11", "2024-10-14",
    "2024-10-15",
])
label_window(demo)

2024-09-20    普通日
2024-09-23    普通日
2024-09-24     节前
2024-09-25     节前
2024-09-26     节前
2024-09-27     节前
2024-09-30     节前
2024-10-08     节后
2024-10-09     节后
2024-10-10     节后
2024-10-11     节后
2024-10-14     节后
2024-10-15    普通日
Name: 窗口, dtype: str